In [2]:
import pandas as pd
import pandera as pa
from pandera import DataFrameSchema, Column, Check
from pandera.dtypes import Float, String, Int, Bool
from pathlib import Path
import warnings
from datetime import datetime
warnings.filterwarnings("ignore")

df = pd.read_csv("dataset1.csv")
df.drop(columns=["Unnamed: 0"], inplace=True)
UNIQUE_TRACK_GENRES = list(df["track_genre"].unique())

def strict_datetime_column(series: pd.Series) -> bool:
    def is_strict_format(val):
        try:
            datetime.strptime(val, "%Y-%m-%d %H:%M:%S UTC")
            return True
        except:
            return False

    mask = series.apply(is_strict_format)
    return mask


music_schema = pa.DataFrameSchema(
    columns={
        "track_id": Column(
            String,
            checks=Check.str_length(22)
        ),
        "artists": Column(
            String,
            checks=Check.str_length(2, 512)
        ),
        "album_name": Column(
            String,
            checks=Check.str_length(2, 512)
        ),
        "track_name": Column(
            String,
            checks=Check.str_length(2, 512)
        ),
        "popularity": Column(
            Int,
            checks=Check.in_range(0, 100)
        ),
        "duration_ms": Column(
            Int,
            checks=Check.in_range(1, 5237760)
        ),
        "explicit": Column(
            Bool
        ),
        "danceability": Column(
            Float,
            checks=Check.in_range(0.0, 1.0)
        ),
        "energy": Column(
            Float,
            checks=Check.in_range(0.0, 1.0)
        ),
        "key": Column(
            Int,
            checks=Check.in_range(0, 11)
        ),
        "loudness": Column(
            Float,
            checks=Check.in_range(-45, 5)
        ),
        "mode": Column(
            Float,
            checks=Check.in_range(0.0, 1.0)
        ),
        "speechiness": Column(
            Float,
            checks=Check.in_range(0.0, 1.0)
        ),
        "acousticness": Column(
            Float,
            checks=Check.in_range(0.0, 1.0)
        ),
        "instrumentalness": Column(
            Float,
            checks=Check.in_range(0.0, 1.0)
        ),
        "liveness": Column(
            Float,
            checks=Check.in_range(0.0, 1.0)
        ),
        "valence": Column(
            Float,
            checks=Check.in_range(0.0, 1.0)
        ),
        "tempo": Column(
            Float,
            checks=Check.in_range(0.0, 256.0)
        ),
        "time_signature": Column(
            Int,
            checks=Check.in_range(0, 5)
        ),
        "track_genre": Column(
            String,
            checks=[
                Check(lambda s: s.nunique() <= 114, error="Wrong number of unique track genres"),
                Check.isin(UNIQUE_TRACK_GENRES, error="Invalid track genre")
            ]
        ),
    },
    strict=True,
    coerce=True,
    ordered=False
)

In [3]:
STUDENT_NAME = "Calestru"
DATETIME = datetime.now().strftime("%Y%m%d")

try:
    music_schema.validate(df, lazy=True)
except pa.errors.SchemaErrors as err:
    errors_df = err.failure_cases
    errors_df["failure_case"] = errors_df["failure_case"].astype(str)
    errors_df.sort_values(by='index', inplace=True)
    errors_df.reset_index(drop=True, inplace=True)
    errors_df.to_parquet(f"{STUDENT_NAME}_{DATETIME}_validation_report.parquet", index=False) 

errors_df

,schema_context,column,check,check_number,failure_case,index
0,Column,album_name,"str_length(2, 512)",0,0,650
1,Column,album_name,"str_length(2, 512)",0,E,910
2,Column,album_name,"str_length(2, 512)",0,V,1456
3,Column,album_name,"str_length(2, 512)",0,4,1629
4,Column,album_name,"str_length(2, 512)",0,&,3911
...,...,...,...,...,...,...
233,Column,album_name,"str_length(2, 512)",0,5,111950
234,Column,album_name,"str_length(2, 512)",0,9,112209
235,Column,album_name,"str_length(2, 512)",0,O,112475
236,Column,album_name,"str_length(2, 512)",0,X,112815


In [4]:
import pandas as pd
from pathlib import Path # для работы независимо от OC Windows\Linux

# Пример: валидация датафрейма
for nrows in [1, 10, 100, 1000]:
    df_timing = pd.read_csv("dataset1.csv", nrows=nrows)
    df_timing.drop(columns=["Unnamed: 0"], inplace=True)
    %timeit music_schema.validate(df_timing, lazy=True)


24 ms ± 9.37 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)
17.4 ms ± 314 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)
17.7 ms ± 152 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


SchemaErrors: {
    "DATA": {
        "DATAFRAME_CHECK": [
            {
                "schema": null,
                "column": "album_name",
                "check": "str_length(2, 512)",
                "error": "Column 'album_name' failed element-wise validator number 0: str_length(2, 512) failure cases: 0, E"
            }
        ]
    }
}